#### STEP 0 - Read JSON 

In [1]:
# ------------------------------------------------------------
# READ FILE
# ------------------------------------------------------------
import json
from pathlib import Path
import pandas as pd

input_path = Path("hard_events.json")

if not input_path.exists():
    raise FileNotFoundError("File not found")

with input_path.open("r", encoding="utf-8") as f:
    raw = json.load(f)

events = raw["data"]   # list of event records


In [2]:
events[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

#### STEP 1 Build Fact Events

In [7]:
# ------------------------------------------------------------
# FACT EVENTS
# ------------------------------------------------------------

fact_events_rows = []

for ev in events:

    user = ev.get("user", {})

    fact_events_rows.append({
        "event_id": ev.get("event_id"),
        "event_time": ev.get("event_time"),
        "event_type": ev.get("event_type"),
        "user_id": pd.to_numeric(user.get("user_id"), errors="coerce")
    })

fact_events = pd.DataFrame(fact_events_rows)

# ----- cleaning -----
fact_events["event_time"] = pd.to_datetime(fact_events["event_time"], utc=True, errors="coerce")
fact_events["user_id"] = fact_events["user_id"].astype("Int64")

fact_events.head()


,event_id,event_time,event_type,user_id
0,ev_9001,2026-02-04 20:00:00+00:00,purchase,301
1,ev_9002,2026-02-04 20:05:00+00:00,click,302


#### STEP 2 Build Dim Users

In [13]:
events[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

In [8]:
# ------------------------------------------------------------
# DIM USERS
# ------------------------------------------------------------

dim_users_rows = []

for ev in events:

    user = ev.get("user", {})
    traits = user.get("traits", {})

    dim_users_rows.append({
        "user_id": pd.to_numeric(user.get("user_id"), errors="coerce"),
        "vip": traits.get("vip"),
        "signup_date": traits.get("signup_date")
    })

dim_users = pd.DataFrame(dim_users_rows)

# ----- cleaning -----
dim_users["user_id"] = dim_users["user_id"].astype("Int64")
dim_users["signup_date"] = pd.to_datetime(dim_users["signup_date"], errors="coerce").dt.date

# IMPORTANT: remove duplicates
dim_users = dim_users.drop_duplicates("user_id").reset_index(drop=True)

dim_users.head()


,user_id,vip,signup_date
0,301,True,2025-11-01
1,302,False,NaT


#### STEP 3 Build Dim Addresses

In [14]:
events[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

In [9]:
# ------------------------------------------------------------
# DIM ADDRESSES
# ------------------------------------------------------------

dim_addresses_rows = []

for ev in events:

    user = ev.get("user", {})
    user_id = pd.to_numeric(user.get("user_id"), errors="coerce")

    for addr in user.get("addresses", []) or []:
        dim_addresses_rows.append({
            "user_id": user_id,
            "address_type": addr.get("type"),
            "city": addr.get("city"),
            "state": addr.get("state")
        })

dim_addresses = pd.DataFrame(dim_addresses_rows)

# ----- cleaning -----
dim_addresses["user_id"] = dim_addresses["user_id"].astype("Int64")
dim_addresses = dim_addresses.drop_duplicates().reset_index(drop=True)

dim_addresses.head()


,user_id,address_type,city,state
0,301,home,Sydney,NSW
1,301,work,Sydney,NSW


#### STEP 4 Build Dim Device

In [15]:
events[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

In [10]:
# ------------------------------------------------------------
# DIM DEVICE
# ------------------------------------------------------------

dim_devices_rows = []

for ev in events:

    payload = ev.get("payload", {})
    device = payload.get("device", {})

    dim_devices_rows.append({
        "event_id": ev.get("event_id"),
        "os": device.get("os"),
        "app_version": device.get("app_version")
    })

dim_devices = pd.DataFrame(dim_devices_rows)

dim_devices.head()


,event_id,os,app_version
0,ev_9001,iOS,5.1.0
1,ev_9002,Android,5.0.3


#### STEP 5 Build Dim Page

In [16]:
events[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

In [11]:
# ------------------------------------------------------------
# DIM PAGE
# ------------------------------------------------------------

dim_pages_rows = []

for ev in events:

    payload = ev.get("payload", {})
    page = payload.get("page")

    if page:
        dim_pages_rows.append({
            "event_id": ev.get("event_id"),
            "url": page.get("url"),
            "referrer": page.get("referrer")
        })

dim_pages = pd.DataFrame(dim_pages_rows)

dim_pages.head()


,event_id,url,referrer
0,ev_9002,/home,None


#### STEP 6 Build Fact_Cart_Items

In [17]:
events[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

In [12]:
# ------------------------------------------------------------
# FACT CART ITEMS
# ------------------------------------------------------------

fact_cart_rows = []

for ev in events:

    if ev.get("event_type") != "purchase":
        continue

    payload = ev.get("payload", {})
    cart = payload.get("cart", {})
    coupon = cart.get("coupon")

    for item in cart.get("items", []) or []:

        qty = pd.to_numeric(item.get("qty"), errors="coerce")
        price = pd.to_numeric(item.get("unit_price"), errors="coerce")

        fact_cart_rows.append({
            "event_id": ev.get("event_id"),
            "sku": item.get("sku"),
            "qty": qty,
            "unit_price": price,
            "line_total": qty * price if pd.notna(qty) and pd.notna(price) else None,
            "coupon": coupon
        })

fact_cart_items = pd.DataFrame(fact_cart_rows)

fact_cart_items.head()


,event_id,sku,qty,unit_price,line_total,coupon
0,ev_9001,X9,1,99.99,99.99,WELCOME10
